# kati 전처리 진행 후 다시 pdf 저장

In [ ]:
import os
import fitz  # PyMuPDF

# 입력 폴더 (KATI PDF만 전처리)
INPUT_DIR = r"C:\Users\kmoon\sesac\00_FinalProject\SESAC-final-2025\data\kang"
OUTPUT_DIR = "./processed_kati_pdf"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 제거 기준 키워드
REMOVE_KEYWORDS = [
    "목차", "contents", "table of contents",
    "저작권", "copyright",
    "disclaimer",
    "참고문헌", "references"
]

def is_noise_page(text: str) -> bool:
    """노이즈(제거 대상 페이지) 판단."""
    # 텍스트가 너무 적은 경우
    if len(text.strip()) < 80:
        return True
    
    # 특정 키워드 포함하는 경우
    lower_text = text.lower()
    for kw in REMOVE_KEYWORDS:
        if kw in lower_text:
            return True
    
    return False


def clean_text(text: str) -> str:
    """기본 텍스트 정제 로직."""
    # 공백 정리, 줄바꿈 통일 등
    t = text.replace("\xa0", " ").replace("\u200b", "")
    t = "\n".join([line.strip() for line in t.split("\n")])
    return t


def preprocess_kati_pdf(pdf_path: str, output_path: str):
    """KATI PDF 전처리 후 새 PDF로 저장."""
    doc = fitz.open(pdf_path)

    # 새 PDF 생성
    new_pdf = fitz.open()  # 빈 PDF

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text = page.get_text("text")

        # 전처리 기준 적용
        if is_noise_page(text):
            continue  # 제거

        cleaned = clean_text(text)

        # 새 페이지 추가
        new_page = new_pdf.new_page()
        new_page.insert_text(
            fitz.Point(40, 50),
            cleaned,
            fontsize=11,
            fontname="helv"
        )

    # 저장
    new_pdf.save(output_path)
    new_pdf.close()
    doc.close()


def run_preprocessing():
    """입력 폴더 내 KATI PDF만 필터링하여 전처리 수행."""
    for filename in os.listdir(INPUT_DIR):
        if not filename.lower().endswith(".pdf"):
            continue
        
        # 파일명에 'kati' 포함된 것만 처리하고 싶은 경우:
        # if "kati" not in filename.lower():
        #     continue

        pdf_path = os.path.join(INPUT_DIR, filename)
        output_path = os.path.join(OUTPUT_DIR, f"cleaned_{filename}")

        print(f"[처리 중] {filename}")
        preprocess_kati_pdf(pdf_path, output_path)
        print(f"[저장 완료] {output_path}\n")


if __name__ == "__main__":
    run_preprocessing()
